In [ ]:
# import libraries
!pip install bitsandbytes

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
import torchvision
import torch.nn.functional as F
from torchvision import transforms
from torchvision.utils import make_grid

from datasets import load_dataset

# diffusion models
from diffusers import DDPMScheduler, DPMSolverMultistepScheduler
from diffusers import UNet2DModel
from diffusers import DDPMPipeline
from diffusers.utils import make_image_grid

# visualization
import matplotlib.pyplot as plt

In [ ]:
data_path = '/kaggle/input/datasets/rijulpaul/ffhq-dataset-128x128/images128'
output_path = '/kaggle/working/'
repo_id = "rijulpaul/ddpm-ffhq128"

device = "cuda" if torch.cuda.is_available() else "cpu"

dataset = load_dataset("imagefolder",data_dir=data_path)['train']

In [ ]:
from dataclasses import dataclass

@dataclass
class TrainingConfig:
    image_size = 128
    batch_size = 32
    eval_batch_size = 8
    num_epochs = 300
    gradient_accumulation_steps = 1
    learning_rate = 1e-4
    lr_warmup_steps = 500
    save_image_epochs = 5
    save_model_epochs = 2
    mixed_precision = "fp16"
    output_dir = output_path

    seed = 0


config = TrainingConfig()

In [ ]:
preprocess = transforms.Compose([
    transforms.Resize((config.image_size,config.image_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.5],[0.5])
])

In [ ]:
def transform(examples):
    images = [preprocess(image.convert("RGB")) for image in examples["image"]]
    return {"images": images}
dataset.set_transform(transform)

dataloader = torch.utils.data.DataLoader(
    dataset,
    batch_size=config.batch_size,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

In [ ]:
# Utility function used to display images
def show_images(image):
    plt.imshow(make_grid(image*0.5+0.5).permute(1,2,0))

In [ ]:
batch=next(iter(dataloader))
plt.figure(figsize=(12,5))
show_images(batch['images'][:4])

In [ ]:
class EMA:
    def __init__(self, model, decay=0.9999, device=None):
        self.decay = decay
        self.device = device

        self.shadow = {}
        self.backup = {}

        for name, param in model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone()

                if self.device:
                    self.shadow[name] = self.shadow[name].to(self.device)

    def update(self, model):
        for name, param in model.named_parameters():
            if param.requires_grad:
                assert name in self.shadow
                new_avg = (1.0 - self.decay) * param.data + self.decay * self.shadow[name]
                self.shadow[name] = new_avg.clone()

    def apply_shadow(self, model):
        self.backup = {}

        for name, param in model.named_parameters():
            if param.requires_grad:
                self.backup[name] = param.data.clone()
                param.data.copy_(self.shadow[name])

    def restore(self, model):
        for name, param in model.named_parameters():
            if param.requires_grad:
                param.data.copy_(self.backup[name])

        self.backup = {}

In [ ]:
# model = UNet2DModel.from_pretrained(repo_id,subfolder="unet",use_safetensors=True)

model = UNet2DModel(
    sample_size=config.image_size,
    in_channels=3,
    out_channels=3,
    layers_per_block=2,
    block_out_channels = (256, 256, 512, 512),
    down_block_types = (
        "DownBlock2D",
        "DownBlock2D",
        "AttnDownBlock2D",
        "DownBlock2D",
    ),
    
    up_block_types = (
        "UpBlock2D",
        "AttnUpBlock2D",
        "UpBlock2D",
        "UpBlock2D",
    )
)

# model.disable_gradient_checkpointing()
model = model.to(device)
model = torch.compile(model)
# model = torch.nn.DataParallel(model)
ema = EMA(model, decay=0.9999)

In [ ]:
# noise_scheduler = DDPMScheduler.from_pretrained(repo_id,subfolder="scheduler")

noise_scheduler = DDPMScheduler(
    num_train_timesteps=1000,
    beta_schedule="squaredcos_cap_v2"
)

In [ ]:
from diffusers.optimization import get_cosine_schedule_with_warmup
import bitsandbytes as bnb

optimizer = bnb.optim.AdamW8bit(model.parameters(), lr=config.learning_rate)

lr_scheduler = get_cosine_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=config.lr_warmup_steps,
    num_training_steps=(len(dataloader) * config.num_epochs),
)

In [ ]:
def evaluate(config, epoch, pipeline, device, model, ema):

    ema.apply_shadow(model)

    pipeline.scheduler = DPMSolverMultistepScheduler.from_config(
        pipeline.scheduler.config
    )

    images = pipeline(
        batch_size=config.eval_batch_size,
        num_inference_steps=50,
        generator=torch.Generator(device=device).manual_seed(42),
    ).images

    ema.restore(model)

    image_grid = make_image_grid(images, rows=2, cols=4)

    test_dir = os.path.join(output_path, "samples")
    os.makedirs(test_dir, exist_ok=True)
    image_grid.save(f"{test_dir}/{epoch:04d}.png")

In [ ]:
import torch
import torch.nn.functional as F
import sys
from tqdm.auto import tqdm
from diffusers import DDPMPipeline

scaler = torch.cuda.amp.GradScaler()
grad_accum_steps = config.gradient_accumulation_steps

for epoch in range(config.num_epochs):

    progress_bar = tqdm(
        total=len(dataloader),
        desc=f"Epoch {epoch}",
        file=sys.stdout,
        leave=True
    )

    for step, batch in enumerate(dataloader):
        clean_images = batch["images"].to(device)

        noise = torch.randn_like(clean_images)
        bs = clean_images.shape[0]

        timesteps = torch.randint(
            0,
            noise_scheduler.config.num_train_timesteps,
            (bs,),
            device=device,
            dtype=torch.int64
        )

        noisy_images = noise_scheduler.add_noise(clean_images, noise, timesteps)

        # ---- forward (fp16) ----
        with torch.cuda.amp.autocast():
            noise_pred = model(
                sample=noisy_images,
                timestep=timesteps,
                return_dict=False
            )[0]
            loss = F.mse_loss(noise_pred, noise)

        # ---- grad accumulation ----
        loss = loss / grad_accum_steps
        scaler.scale(loss).backward()

        progress_bar.set_postfix({
            "loss": loss.item() * grad_accum_steps
        })

        # if step % 50 == 0:
        #     print(f"Epoch {epoch} | Step {step} | Loss: {loss.item() * grad_accum_steps:.4f}")

        if (step + 1) % grad_accum_steps == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            lr_scheduler.step()

            ema.update(model)

        progress_bar.update(1)

    pipeline = DDPMPipeline(
        unet=model,
        scheduler=noise_scheduler
    )

    if (epoch + 1) % config.save_image_epochs == 0 or epoch == config.num_epochs - 1:
        evaluate(config, epoch, pipeline, device, model, ema)

    if (epoch + 1) % config.save_model_epochs == 0 or epoch == config.num_epochs - 1:
        checkpoint = {
            "model": model.state_dict(),  # DataParallel
            "optimizer": optimizer.state_dict(),
            "lr_scheduler": lr_scheduler.state_dict(),
            "scaler": scaler.state_dict(),
            "ema": ema.shadow,
            "epoch": epoch,
            "step": step,
        }

        torch.save(checkpoint, os.path.join(config.output_dir, "training_checkpoint.pt"))
        pipeline.save_pretrained(
            os.path.join(config.output_dir, "checkpoint")
        )

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
import glob
from PIL import Image
import numpy as np

fig = plt.figure(figsize=(8,8))
plt.axis("off")

ims = []
for path in sorted(glob.glob(f"{config.output_dir}/samples/*.png")):
    img = np.array(Image.open(path))
    ims.append([plt.imshow(img, animated=True)])

ani = animation.ArtistAnimation(
    fig, ims, interval=400, repeat_delay=1000, blit=True
)

HTML(ani.to_jshtml())

In [ ]:
import glob
from PIL import Image

sample_images = sorted(glob.glob(f"{output_path}/samples/*.png"))
Image.open(sample_images[-1])